In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='f327fb39-c703-423d-a33b-b9195fe387ea'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Searching for langchain-mcp-adapters**\n\nI need to provide information about the langchain-mcp-adapters library, which seems to be part of the LangChain ecosystem. It might relate to adapters for MCP, though I’m unsure. I can use the search_web tool to dig deeper, and I wonder if there’s a GitHub repository linked to this. The available tools are somewhat confusing since it mentions github_file, but only search_web is accessible right now. So, I’ll proceed with that to see what I can find!**Searching for langchain-mcp-adapters**\n\nI think the langchain-mcp-adapters might be related to bridging LangChain with something called MCP, possibly the Minecraft Protocol? But there are other interpretations too, like Model-Compiler-Protocol or 

## Online MCP

In [15]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [16]:
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=tools,
)

In [17]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='4c5ba43c-0ec0-4b00-b372-899270d867fd'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Fetching current time**\n\nI need to respond to the user asking, "What time is it?" Since the user didn\'t specify a timezone, I\'ll default to America/New_York. I\'ll use the get_current_time tool to fetch the time in that timezone and present it in a simple format. It might also be good to ask if they need a different timezone to clarify. So I’ll get started on that tool call and then handle the response!**Using the time tool**\n\nI\'m ready to call the get_current_time tool, using the timezone "America/New_York." After that, I’ll present the current date and time to the user. The tool should return a time string and a date, likely in a typical format like { "time": "14:23:45", "date": "2026-08-25" }. I can\'t assume the exact format, so I\'ll check what it returns 